In [ ]:
from platon.transit_depth_calculator import TransitDepthCalculator
from platon.constants import M_jup, R_jup, R_sun, M_earth, R_earth, G
from astropy.constants import R
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import interpolate
import scipy
from spectres import spectres
import pickle
import arviz as az
import corner

In [ ]:
def readInDataCSV(fn):
    infile = open(fn,'r')
    lines = infile.readlines()[1:]
    infile.close()
    nl = len(lines)
    wv_,td_,tderr_ = np.zeros(nl),np.zeros(nl),np.zeros(nl)
    wvbins_ = np.zeros((nl,2))
    for i in range(nl):
        line = lines[i].split(',')
        wv_[i] = float(line[0])/1e6 # m
        wvbins_[i,0] = float(line[1])/1e6 # m
        wvbins_[i,1] = float(line[2])/1e6 # m
        td_[i] = float(line[3])**2 # Rp/Rs -> transit depth 
        tderr_[i] = 2.*float(line[3])*float(line[4]) # Rp/Rs error -> transit depth error
    return wv_,wvbins_,td_,tderr_,nl

In [ ]:
# Read and concatenate data

wv141,wvbins141,td141,tderr141,n141 = readInDataCSV('aumicb_data/F21-contam-blind-Dec28.csv')
wv102,wvbins102,td102,tderr102,n102 = readInDataCSV('aumicb_data/S22-contam-blind-Dec28.csv')
wv = np.concatenate((wv102,wv141))
wvbins = np.concatenate((wvbins102,wvbins141))
td = np.concatenate((td102,td141))
tderr = np.concatenate((tderr102,tderr141))
print(wv,td,tderr)

In [ ]:
# Plot settings
plt.rc('font', family='sans-serif')
fontname = 'sans-serif'
labelpad=5

# from toi260.01.mplstyle

plt.rc('lines', linewidth=2.7)              
plt.rc('lines', markerfacecolor='w')
plt.rc('lines', markeredgewidth=2.0)          
plt.rc('lines', markersize=8)  

plt.rc('font', size=17)

plt.rc('text', usetex=False)

plt.rc('axes', labelsize = 17)
plt.rc('axes', linewidth=2)
plt.rc('axes', labelweight='normal')
plt.rc('axes', axisbelow=False)

plt.rc('xtick', top=True)     
plt.rc('xtick', bottom=True)     
plt.rc('xtick.major', size=6)
plt.rc('xtick.minor', size=4)      
plt.rc('xtick.major', width=2)     
plt.rc('xtick.minor', width=2)    
plt.rc('xtick', labelsize=14)      
plt.rc('xtick', direction='in')      
plt.rc('xtick.minor', visible=True)    
plt.rc('xtick.major', top=True)    
plt.rc('xtick.major', bottom=True)    
plt.rc('xtick.minor', top=True)    
plt.rc('xtick.minor', bottom=True)   

plt.rc('ytick', left=True)     
plt.rc('ytick', right=True)     
plt.rc('ytick.major', size=6)
plt.rc('ytick.minor', size=4)      
plt.rc('ytick.major', width=2)     
plt.rc('ytick.minor', width=2)    
plt.rc('ytick', labelsize=14)      
plt.rc('ytick', direction='in')      
plt.rc('ytick.minor', visible=True)    
plt.rc('ytick.major', left=True)    
plt.rc('ytick.major', right=True)    
plt.rc('ytick.minor', left=True)    
plt.rc('ytick.minor', right=True)  
  
plt.rc('legend', fontsize=18)

plt.rc('image', aspect='auto')        
plt.rc('image', origin='lower')        

plt.rc('errorbar', capsize=0)

plt.rc('savefig', bbox='tight')

In [ ]:
fname = 'aumicb_retrieval_results/aumicb_nlive1000_contam_newdataDec28_errinfl_widerMpPrior_newT.pkl'
with open(fname,'rb') as f:
    out = pickle.load(f)
print(out.keys())
print(out['labels'])
print(out['best_fit_params'])
print(out['logZ'])

out.equal_samples[:,2] = out.equal_samples[:,2] / M_earth
out.equal_samples[:,3] = out.equal_samples[:,3] / R_earth
out.equal_samples[:,9] = out.equal_samples[:,9] - 5
out.equal_samples[:,10] = out.equal_samples[:,10] * 1e6
out.equal_samples[:,11] = out.equal_samples[:,11] * 1e6

print('')

fnametls = 'aumicb_retrieval_results/aumicb_nlive1000_contam_newdataDec28_errinfl_TLSonly_newT.pkl'
with open(fnametls,'rb') as f:
    outtls = pickle.load(f)
print(outtls.keys())
print(outtls['labels'])
print(outtls['best_fit_params'])
print(outtls['logZ'])

print('')

fnameflat = 'aumicb_retrieval_results/aumicb_nlive1000_contam_newdataDec28_errinfl_flatline.pkl'
with open(fnameflat,'rb') as f:
    outflat = pickle.load(f)
print(outflat.keys())
print(outflat['labels'])
print(outflat['best_fit_params'])
print(outflat['logZ'])

fnameclear = 'aumicb_retrieval_results/aumicb_nlive1000_contam_newdataDec28_errinfl_widerMpPrior_newT_clear.pkl'
with open(fnameclear,'rb') as f:
    outclear = pickle.load(f)
print(outclear.keys())
print(outclear['labels'])
print(outclear['best_fit_params'])
print(outclear['logZ'])

In [ ]:
out['best_fit_transit_dict'].keys()

In [ ]:
calculator = TransitDepthCalculator()

In [ ]:
spectout = {}
numsamp = min(1000,len(out['equal_samples'][:,0]))
sharr,pcldarr,scatfarr = np.zeros(numsamp),np.zeros(numsamp),np.zeros(numsamp)
#wavelengths_list,depths_list,info_dict_list,binwv_list,bintd_list = [],[],[],[],[]
depths_list,bintd_list,bintd_full_list = [],[],[]
for i in range(numsamp):
    print('Regenerating spectra',i)
    params = out['equal_samples'][i,:]
    datawv = out['transit_wavelengths']
    #vmrs=[]
    #for j in range(len(species)-1):
    #    vmrs.append(10.**params[j])
    #vmrs.append(1.-np.sum(vmrs))
    Rs = 0.8*R_sun
    Mp = params[2] * M_earth
    Rp = params[3] * R_earth
    T = params[4]
    logZ = params[8]
    CO_ratio = params[12]
    wavelengths, depths, info_dict = calculator.compute_depths(Rs, Mp, Rp, T, logZ=logZ, CO_ratio=CO_ratio, 
                                                               #gases=species,vmrs=vmrs, 
                                                               scattering_factor=10.**params[5],
                                                               scattering_slope=params[6],
                                                               cloudtop_pressure=10.**(params[9]+5),
                                                               T_star=params[0],T_spot=params[1],
                                                               spot_cov_frac=params[7],full_output=True)
    
    # compute scale heights 
    #print(info_dict['P_profile'],10.**(params[9]+5))
    if np.max(info_dict['P_profile']) > 1e5:
        #print(len(info_dict['P_profile']),len(info_dict['mu_profile']))
        mu = interpolate.interp1d(info_dict['P_profile'],
                                  info_dict['mu_profile'][:len(info_dict['P_profile'])])(1e5)/1e3
    else:
        mu = info_dict['mu_profile'][-1]/1e3
    #print(mu)
    gr = G*Mp/Rp**2
    sh = R.value*T/(mu*gr)
    print(Mp,Rp,mu,gr,sh)
    sharr[i] = sh
    pcldarr[i] = 10.**(params[9]+5)
    scatfarr[i] = 10.**params[5]
        
    depths[0:len(wavelengths[wavelengths<datawv[n102-1]])] += params[11] / 1e6
    
    binwv = np.linspace(np.min(datawv),np.max(datawv),num=100)    
    bintd = spectres(binwv, wavelengths, depths)
        
    binwv_full = np.logspace(np.log10(0.3e-6),np.log10(15e-6),num=1000)
    bintd_full = spectres(binwv_full,wavelengths,depths)
    
    #print(bintd_full)

    if i%50 == 0:
        plt.semilogx(wavelengths*1e6,depths)
        plt.semilogx(binwv*1e6,bintd)
        plt.semilogx(binwv_full*1e6,bintd_full)
        plt.semilogx(out['transit_wavelengths']*1e6,out['random_transit_depths'][i])
        plt.xlim([0.5,15.0])
        plt.show()        

    #wavelengths_list.append(wavelengths)
    depths_list.append(depths)
    #info_dict_list.append(info_dict)
    #binwv_list.append(binwv)
    bintd_list.append(bintd)
    bintd_full_list.append(bintd_full)

spectout['wavelengths'] = wavelengths
spectout['depths'] = depths_list
#spectout['info_dict'] = info_dict_list
spectout['binwv'] = binwv
spectout['bintd'] = bintd_list
spectout['binwv_full'] = binwv_full
spectout['bintd_full'] = bintd_full_list

f = open('aumicb_newDataDec28_newT_100PointRandomModelSpectra.pkl','wb')
pickle.dump(spectout,f)
f.close()

In [ ]:
spectouttls = {}
numsamp = min(1000,len(outtls['equal_samples'][:,0]))
#wavelengths_list,depths_list,info_dict_list,binwv_list,bintd_list = [],[],[],[],[]
depths_list,bintd_list,bintd_full_list = [],[],[]
for i in range(numsamp):
    print('Regenerating spectra',i)
    params = outtls['equal_samples'][i,:]
    datawv = outtls['transit_wavelengths']
    #vmrs=[]
    #for j in range(len(species)-1):
    #    vmrs.append(10.**params[j])
    #vmrs.append(1.-np.sum(vmrs))
    Rs = 0.8*R_sun
    Mp = 50.*M_earth
    Rp = params[2]
    T = 200.0
    logZ = -1.
    CO_ratio = 0.59
    wavelengths, depths, info_dict = calculator.compute_depths(Rs, Mp, Rp, T, logZ=logZ, CO_ratio=CO_ratio, 
                                                               #gases=species,vmrs=vmrs, 
                                                               scattering_factor=1.,
                                                               scattering_slope=4.,
                                                               cloudtop_pressure=10.**-3.999,
                                                               T_star=params[0],T_spot=params[1],
                                                               spot_cov_frac=params[3],full_output=True)
        
    depths[0:len(wavelengths[wavelengths<datawv[n102-1]])] += params[5]
    
    binwv = np.linspace(np.min(datawv),np.max(datawv),num=100)    
    bintd = spectres(binwv,wavelengths,depths)
    
    binwv_full = np.logspace(np.log10(0.3e-6),np.log10(15e-6),num=1000)
    bintd_full = spectres(binwv_full,wavelengths,depths)

    if i%50 == 0:
        plt.semilogx(wavelengths*1e6,depths)
        plt.semilogx(binwv*1e6,bintd)
        plt.semilogx(binwv_full*1e6,bintd_full)
        plt.semilogx(outtls['transit_wavelengths']*1e6,outtls['random_transit_depths'][i])
        plt.xlim([0.5,15.0])
        plt.show()        

    #wavelengths_list.append(wavelengths)
    depths_list.append(depths)
    #info_dict_list.append(info_dict)
    #binwv_list.append(binwv)
    bintd_list.append(bintd)
    bintd_full_list.append(bintd_full)

spectouttls['wavelengths'] = wavelengths
spectouttls['depths'] = depths_list
#spectouttls['info_dict'] = info_dict_list
spectouttls['binwv'] = binwv
spectouttls['bintd'] = bintd_list
spectouttls['binwv_full'] = binwv_full
spectouttls['bintd_full'] = bintd_full_list

f = open('aumicb_newDataDec28_TLSonly_newT_100PointRandomModelSpectra.pkl','wb')
pickle.dump(spectouttls,f)
f.close()

In [ ]:
spectoutclear = {}
numsamp = min(1000,len(outclear['equal_samples'][:,0]))
shcleararr = np.zeros(numsamp)
#wavelengths_list,depths_list,info_dict_list,binwv_list,bintd_list = [],[],[],[],[]
depths_list,bintd_list,bintd_full_list = [],[],[]
for i in range(numsamp):
    print('Regenerating spectra',i)
    params = outclear['equal_samples'][i,:]
    datawv = outclear['transit_wavelengths']
    #vmrs=[]
    #for j in range(len(species)-1):
    #    vmrs.append(10.**params[j])
    #vmrs.append(1.-np.sum(vmrs))
    Rs = 0.8*R_sun
    Mp = params[2] 
    Rp = params[3] 
    T = params[4]
    logZ = params[6]
    CO_ratio = params[9]
    wavelengths, depths, info_dict = calculator.compute_depths(Rs, Mp, Rp, T, logZ=logZ, CO_ratio=CO_ratio, 
                                                               #gases=species,vmrs=vmrs, 
                                                               scattering_factor=1,
                                                               scattering_slope=4,
                                                               cloudtop_pressure=np.inf,
                                                               T_star=params[0],T_spot=params[1],
                                                               spot_cov_frac=params[5],full_output=True)
    
    # compute scale heights 
    #print(info_dict['P_profile'],10.**(params[9]+5))
    if np.max(info_dict['P_profile']) > 1e5:
        #print(len(info_dict['P_profile']),len(info_dict['mu_profile']))
        mu = interpolate.interp1d(info_dict['P_profile'],
                                  info_dict['mu_profile'][:len(info_dict['P_profile'])])(1e5)/1e3
    else:
        mu = info_dict['mu_profile'][-1]/1e3
    #print(mu)
    gr = G*Mp/Rp**2
    sh = R.value*T/(mu*gr)
    print(Mp,Rp,mu,gr,sh)
    shcleararr[i] = sh
        
    depths[0:len(wavelengths[wavelengths<datawv[n102-1]])] += params[8] 
    
    binwv = np.linspace(np.min(datawv),np.max(datawv),num=100)    
    bintd = spectres(binwv, wavelengths, depths)
        
    binwv_full = np.logspace(np.log10(0.3e-6),np.log10(15e-6),num=1000)
    bintd_full = spectres(binwv_full,wavelengths,depths)
    
    #print(bintd_full)

    if i%50 == 0:
        plt.semilogx(wavelengths*1e6,depths)
        plt.semilogx(binwv*1e6,bintd)
        plt.semilogx(binwv_full*1e6,bintd_full)
        plt.semilogx(outclear['transit_wavelengths']*1e6,outclear['random_transit_depths'][i])
        plt.xlim([0.5,15.0])
        plt.show()        

    #wavelengths_list.append(wavelengths)
    depths_list.append(depths)
    #info_dict_list.append(info_dict)
    #binwv_list.append(binwv)
    bintd_list.append(bintd)
    bintd_full_list.append(bintd_full)

spectoutclear['wavelengths'] = wavelengths
spectoutclear['depths'] = depths_list
#spectout['info_dict'] = info_dict_list
spectoutclear['binwv'] = binwv
spectoutclear['bintd'] = bintd_list
spectoutclear['binwv_full'] = binwv_full
spectoutclear['bintd_full'] = bintd_full_list

f = open('aumicb_newDataDec28_newT_clear_100PointRandomModelSpectra.pkl','wb')
pickle.dump(spectoutclear,f)
f.close()

In [ ]:
fname = 'aumicb_newDataDec28_newT_100PointRandomModelSpectra.pkl'
with open(fname,'rb') as f:
    spec = pickle.load(f)
    
fnametls = 'aumicb_newDataDec28_TLSonly_newT_100PointRandomModelSpectra.pkl'
with open(fnametls,'rb') as f:
    spectls = pickle.load(f)

In [ ]:
spec['info_dict']

In [ ]:
fig, axs = plt.subplots(nrows = 2, ncols = 1, figsize = (10,10), facecolor='w')
ax = axs[0]
ax1 = axs[1]
ax.errorbar(wv*1e6,td*1e2,yerr=tderr*1e2,
            ls='',color='k',marker='o',zorder=3)
#ax.set_xlabel('Wavelength ($\mu$m)')
ax.set_ylabel('Transit Depth (%)')  
ax.set_ylim([0.215,0.285])
ax1.errorbar(wv*1e6,td*1e2,yerr=tderr*1e2,
            ls='',color='k',marker='o',zorder=3,
            markersize=3,markeredgewidth=0.5,elinewidth=0.5)
ax1.set_xlabel('Wavelength ($\mu$m)')
ax1.set_ylabel('Transit Depth (%)')  
ax1.set_xlim([0.3,12])
ax1.set_ylim([0.195,0.34])

#ax1.set_xticklabels([])
#ax1.set_ylim([0.215,0.285])
#ax.set_ylim([1.1,1.16])


lower_spectrum_1sig = np.percentile(spec['bintd'], 15.865, axis=0)
upper_spectrum_1sig = np.percentile(spec['bintd'], 84.135, axis=0)
#lower_spectrum_2sig = np.percentile(spec['bintd'], 2.275, axis=0)
#upper_spectrum_2sig = np.percentile(spec['bintd'], 97.725, axis=0)
#lower_spectrum_3sig = np.percentile(spec['bintd'], 0.135, axis=0)
#upper_spectrum_3sig = np.percentile(spec['bintd'], 99.865, axis=0)

lower_spectrum_tls_1sig = np.percentile(spectls['bintd'], 15.865, axis=0)
upper_spectrum_tls_1sig = np.percentile(spectls['bintd'], 84.135, axis=0)
#lower_spectrum_tls_2sig = np.percentile(spectls['bintd'], 2.275, axis=0)
#upper_spectrum_tls_2sig = np.percentile(spectls['bintd'], 97.725, axis=0)
#lower_spectrum_tls_3sig = np.percentile(spectls['bintd'], 0.135, axis=0)
#upper_spectrum_tls_3sig = np.percentile(spectls['bintd'], 99.865, axis=0)

lower_spectrum_full_1sig = np.percentile(spec['bintd_full'], 15.865, axis=0)
upper_spectrum_full_1sig = np.percentile(spec['bintd_full'], 84.135, axis=0)

lower_spectrum_tls_full_1sig = np.percentile(spectls['bintd_full'], 15.865, axis=0)
upper_spectrum_tls_full_1sig = np.percentile(spectls['bintd_full'], 84.135, axis=0)


#ax.fill_between(spec['binwv']*1e6,
#                 lower_spectrum_3sig*1e2,
#                 upper_spectrum_3sig*1e2,
#                 color='xkcd:azure',zorder=0,alpha=0.2)
#ax.fill_between(spec['binwv']*1e6,
#                 lower_spectrum_2sig*1e2,
#                 upper_spectrum_2sig*1e2,
#                 color='xkcd:azure',zorder=1,alpha=0.5)
ax.fill_between(spec['binwv']*1e6,
                 lower_spectrum_1sig*1e2,
                 upper_spectrum_1sig*1e2,
                 color='xkcd:azure',zorder=2,alpha=0.7)


#ax.fill_between(spectls['binwv']*1e6,
#                 lower_spectrum_tls_3sig*1e2,
#                 upper_spectrum_tls_3sig*1e2,
#                 color='xkcd:tomato',zorder=0,alpha=0.2)
#ax.fill_between(spectls['binwv']*1e6,
#                 lower_spectrum_tls_2sig*1e2,
#                 upper_spectrum_tls_2sig*1e2,
#                 color='xkcd:tomato',zorder=1,alpha=0.5)
ax.fill_between(spectls['binwv']*1e6,
                 lower_spectrum_tls_1sig*1e2,
                 upper_spectrum_tls_1sig*1e2,
                 color='xkcd:vermillion',zorder=2,alpha=0.7)

ax1.fill_between(spec['binwv_full']*1e6,
                 lower_spectrum_full_1sig*1e2,
                 upper_spectrum_full_1sig*1e2,
                 color='xkcd:azure',zorder=2,alpha=0.7)

ax1.fill_between(spectls['binwv_full']*1e6,
                 lower_spectrum_tls_full_1sig*1e2,
                 upper_spectrum_tls_full_1sig*1e2,
                 color='xkcd:vermillion',zorder=2,alpha=0.7)

ax1.axvspan(2.432,4.013,edgecolor=None,facecolor='Gold',alpha=0.2)
ax1.axvspan(3.881,4.982,edgecolor=None,facecolor='Purple',alpha=0.2)


ax1.set_xscale('log')
ax1.set_xticks([0.3,0.5,0.7,1,2,3,4,5,8,10])
ax1.set_xticklabels([0.3,0.5,0.7,1,2,3,4,5,8,10])

ax.plot([1.3,1.6],[0.262-0.005,0.262-0.005],color='k')
#ax.plot([0.8,1.2],[0.287,0.287],color='k')

ax.annotate('H$_2$O + CH$_4$',xy=(1,0.25),xytext=(0.5*(1.3+1.6),0.263-0.005),color='k',fontsize=12,
             verticalalignment='bottom',horizontalalignment='center') 
#ax.annotate('TLS',xy=(1,0.25),xytext=(1.0,0.288),color='k',fontsize=12,
#             verticalalignment='bottom',horizontalalignment='center') 

ax.annotate('Full Model',xy=(1,0.25),xytext=(1.6,0.274),color='xkcd:azure',fontsize=15,
             verticalalignment='bottom',horizontalalignment='right') 
ax.annotate('TLS Only',xy=(1,0.25),xytext=(1.6,0.268),color='xkcd:vermillion',fontsize=15,
             verticalalignment='bottom',horizontalalignment='right') 

ax1.annotate('HST',xy=(1,0.25),xytext=(1e6*np.sqrt(np.min(spec['binwv'])*np.max(spec['binwv'])),0.27),
            color='k',fontsize=15,verticalalignment='bottom',horizontalalignment='center') 
ax1.annotate('JWST',xy=(1,0.25),xytext=(np.sqrt(2.432*4.958),0.323),
            color='k',fontsize=15,verticalalignment='bottom',horizontalalignment='center') 
ax1.annotate('NIRCam',xy=(1,0.25),xytext=(np.sqrt(2.432*4.958),0.31),
            color='k',fontsize=15,verticalalignment='bottom',horizontalalignment='center') 
ax1.annotate('F322W2',xy=(1,0.25),xytext=(np.sqrt(2.432*4.013),0.275),rotation=90,
            color='xkcd:goldenrod',fontsize=15,verticalalignment='center',horizontalalignment='center') 
ax1.annotate('F444W',xy=(1,0.25),xytext=(np.sqrt(3.881*4.982),0.275),rotation=90,
            color='Purple',fontsize=15,verticalalignment='center',horizontalalignment='center') 

plt.savefig('fig_newDataDec28_newT_retrieval.png', format = 'png')
plt.savefig('fig_newDataDec28_newT_retrieval.pdf', format = 'pdf')
plt.close() 

# corner plot 

print(out.equal_samples[0,:])
pnames = ['T$_{\\rm p}$ (K)','T$_{\\rm s}$ (K)','M$_{\\rm p}$ (M$_{\oplus}$)',
          'R$_{\\rm p}$ (R$_{\oplus}$)','T (K)','log(f)',
          '$m_{\\rm s}$','f$_{\\rm s}$','${\\rm [M/H]}$ ($\\times$ Solar)',
          'log(P$_{\\rm c}$) (bar)','$\epsilon^{\prime}$ (ppm)','$\Delta D$ (ppm)','C/O']
titles = ['T$_{\\rm p}$','T$_{\\rm s}$','M$_{\\rm p}$','R$_{\\rm p}$',
          'T','logf','$m_{\\rm s}$','f$_{\\rm s}$',
          '${\\rm [M/H]}$','logP$_{\\rm c}$','$\epsilon^{\prime}$','$\Delta D$','C/O']
#titlefmt = ['.0f','.0f','.0f','.0f','.0f','.0f','.0f','.2f','.0f','.0f','.0f','.0f','.0f']

fig = plt.figure(figsize=(26,26))
fig1 = corner.corner(out.equal_samples,
                    range=[0.99] * out.equal_samples.shape[1],
                    show_titles=True, levels=[0.393, 0.865, 0.989],
                    #labels=out.fit_info.fit_param_names,
                    labels=pnames,fig=fig,#title_fmt=titlefmt,
                    titles=titles,color='xkcd:deep sea blue',
                    title_kwargs={'pad':0.5,'fontdict':{'fontsize':15}},
                    quantiles=[0.16,0.5,0.86])

fig1.savefig('fig_platoncorner.png',facecolor='w')
fig1.savefig('fig_platoncorner.pdf',facecolor='w')
plt.close()



In [ ]:
# scale height vs. cloud top pressure
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (5,5), facecolor='w')

ct = az.plot_kde(sharr/1e3,np.log10(pcldarr),
                 hdi_probs=[0.393, 0.865, 0.989],ax=ax,
                 contourf_kwargs={'colors': ['w','xkcd:azure','xkcd:cerulean','xkcd:ocean blue'], 'alpha': 1},
                 contour_kwargs={'colors': ['xkcd:ocean blue','xkcd:azure','xkcd:cerulean']})

#ct = az.plot_kde(np.log10(mettot5me/M0),ctoo5me,
#                 hdi_probs=[0.393, 0.865, 0.989],ax=ax2,
#                 contourf_kwargs={'colors': colors[ii], 'alpha': 0.3},
#                 contour_kwargs={'colors': colors[ii]}) 

#ax2.annotate(annotes[ii],xy=(1,0.5),xytext=(-0.7,0.45-ii*0.05),color=colors[ii],horizontalalignment='left') 

#ax.axhline(0.59,color='k',linestyle='--')
#ax.axvline(0,color='gray',linestyle='--')
#ax2.axhline(0.59,color='gray',linestyle='--')

#ax1.set_ylim([0,0.65])
#ax.set_xlim([0,3])

#ax1.annotate('Outside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 
#ax2.annotate('Inside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 

#ax.annotate('Solar C/O',xy=(0,0.5),xytext=(1.25,0.63),color='k',
#             horizontalalignment='center',fontsize=12) 
#ax2.annotate('Solar C/O = 0.59 (Asplund et al. 2021)',xy=(0,0.5),xytext=(1,0.56),color='gray',
#             horizontalalignment='center') 

ax.set_xlabel('Scale Height (km)')
ax.set_ylabel('Cloud Top Pressure')

plt.savefig('fig_newT_shpcld.png', format = 'png')
plt.savefig('fig_newT_shpcld.pdf', format = 'pdf')
plt.close()

In [ ]:
# scale height vs. cloud top pressure
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (5,5), facecolor='w')

print(np.max(sharr),np.min(sharr))

ct = az.plot_dist(sharr/1e3,color='xkcd:azure',ax=ax,quantiles=[0.5])

ax.axvline(np.median(sharr/1e3),color='k',linestyle='--')
#ax.axvline(np.percentile(sharr/1e3, 15.865),color='k',linestyle=':')
#ax.axvline(np.percentile(sharr/1e3, 84.135),color='k',linestyle=':')
ax.axvline(np.percentile(sharr/1e3, 99.865),color='Tomato',linestyle=':')

#print(np.median(sharr/1e3),np.median(sharr/1e3)-np.percentile(sharr/1e3, 15.865),
#      np.percentile(sharr/1e3, 84.135)-np.median(sharr/1e3))

print(np.median(sharr/1e3),np.percentile(sharr/1e3, 99.865))

ax.set_xlim([10,192])
ax.set_ylim([0,0.015])

#ax1.annotate('Outside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 
#ax2.annotate('Inside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 

#ax.set_title('H = 52$^{+38}_{-27}$ km',color='k',pad=10) 
#ax2.annotate('Solar C/O = 0.59 (Asplund et al. 2021)',xy=(0,0.5),xytext=(1,0.56),color='gray',
#             horizontalalignment='center') 

ax.annotate('Median',xy=(50,0),xytext=(45,0.006),color='k',
             horizontalalignment='center',verticalalignment='center',rotation=90) 
ax.annotate('3$\sigma$',xy=(50,0),xytext=(179,0.006),color='Tomato',
             horizontalalignment='center',verticalalignment='center',rotation=90) 

ax.set_xlabel('Scale Height (km)')
ax.set_ylabel('PDF')

plt.savefig('fig_newT_sh.png', format = 'png')
plt.savefig('fig_newT_sh.pdf', format = 'pdf')
plt.close()

In [ ]:
# metallicity vs. C/O
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (5,5), facecolor='w')

ct = az.plot_kde(out['equal_samples'][:,8],out['equal_samples'][:,12],
                 hdi_probs=[0.393, 0.865, 0.989],ax=ax,
                 contourf_kwargs={'colors': ['w','xkcd:azure','xkcd:cerulean','xkcd:ocean blue'], 'alpha': 1},
                 contour_kwargs={'colors': ['xkcd:ocean blue','xkcd:azure','xkcd:cerulean']})

#ct = az.plot_kde(np.log10(mettot5me/M0),ctoo5me,
#                 hdi_probs=[0.393, 0.865, 0.989],ax=ax2,
#                 contourf_kwargs={'colors': colors[ii], 'alpha': 0.3},
#                 contour_kwargs={'colors': colors[ii]}) 

#ax2.annotate(annotes[ii],xy=(1,0.5),xytext=(-0.7,0.45-ii*0.05),color=colors[ii],horizontalalignment='left') 

ax.axhline(0.59,color='k',linestyle='--')
#ax.axvline(0,color='gray',linestyle='--')
#ax2.axhline(0.59,color='gray',linestyle='--')

#ax1.set_ylim([0,0.65])
ax.set_xlim([0,3])

#ax1.annotate('Outside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 
#ax2.annotate('Inside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 

ax.annotate('Solar C/O',xy=(0,0.5),xytext=(1.25,0.63),color='k',
             horizontalalignment='center',fontsize=12) 
#ax2.annotate('Solar C/O = 0.59 (Asplund et al. 2021)',xy=(0,0.5),xytext=(1,0.56),color='gray',
#             horizontalalignment='center') 

ax.set_ylabel('C/O')
ax.set_xlabel('[M/H]')

plt.savefig('fig_newT_metc2oposteriors.png', format = 'png')
plt.savefig('fig_newT_metc2oposteriors.pdf', format = 'pdf')
plt.close()

In [ ]:
def readInDataEur(fn):
    infile = open(fn,'r')
    lines = infile.readlines()[1:]
    infile.close()
    nl = len(lines)
    wv_,td_,tderr_ = np.zeros(nl),np.zeros(nl),np.zeros(nl)
    wvbins_ = np.zeros((nl,2))
    for i in range(nl):
        line = lines[i].split(',')
        wv_[i] = round(float(line[0]),3)/1e6 # micron -> m, round to 3 decimal places to remove numerical stuff
        binhalfwid = round(float(line[1]),3)/1e6 # micron -> m, round to 3 decimal places to remove numerical stuff
        wvbins_[i,0] = wv_[i]-binhalfwid # m
        wvbins_[i,1] = wv_[i]+binhalfwid # m
        td_[i] = float(line[2]) # transit depth 
        tderr_[i] = max(float(line[3]),float(line[4])) # larger of the asymmetric transit depth errors (conservative option)
    nnrs1_ = 0
    for i in range(nl-1):
        if wv_[i+1]-wv_[i] < binhalfwid*3.:
            nnrs1_ += 1
        else:
            nnrs1_final = nnrs1_
    return wv_,wvbins_,td_,tderr_,nnrs1_final

def readInDataTib(fn):
    infile = open(fn,'r')
    lines = infile.readlines()[1:]
    infile.close()
    nl = len(lines)
    wv_,td_,tderr_ = np.zeros(nl),np.zeros(nl),np.zeros(nl)
    wvbins_ = np.zeros((nl,2))
    for i in range(nl):
        line = lines[i].split()
        wv_[i] = float(line[0])/1e6 # micron -> m
        binfullwid = float(line[1])/1e6 # micron -> m
        wvbins_[i,0] = wv_[i]-binfullwid/2.
        wvbins_[i,1] = wv_[i]+binfullwid/2.
        td_[i] = float(line[2])/1e6 # ppm -> nothing 
        tderr_[i] = float(line[3])/1e6 # ppm -> nothing 
    nnrs1_ = 0
    for i in range(nl-1):
        if wv_[i+1]-wv_[i] < binfullwid*1.5:
            nnrs1_ += 1
        else:
            nnrs1_final = nnrs1_
    return wv_,wvbins_,td_,tderr_,nnrs1_final

def readInDataTsw(fn):
    infile = open(fn,'r')
    lines = infile.readlines()[1:]
    infile.close()
    nl = len(lines)
    wv_,td_,tderr_ = np.zeros(nl),np.zeros(nl),np.zeros(nl)
    wvbins_ = np.zeros((nl,2))
    for i in range(nl):
        line = lines[i].split(',')
        wv_[i] = float(line[0])/1e6 # micron -> m
        td_[i] = float(line[1])/1e6 # ppm -> nothing 
        tderr_[i] = float(line[2])/1e6 # ppm -> nothing 
    binhalfwid = (wv_[1]-wv_[0])/2. # assuming bin width is constant with wavelength
    for i in range(nl):
        wvbins_[i,0] = wv_[i]-binhalfwid
        wvbins_[i,1] = wv_[i]+binhalfwid
    nnrs1_ = 0
    for i in range(nl-1):
        if wv_[i+1]-wv_[i] < binhalfwid*3.:
            nnrs1_ += 1
        else:
            nnrs1_final = nnrs1_
    return wv_,wvbins_,td_,tderr_,nnrs1_final

In [ ]:
# Read in data
dfiles,wv,wvbins,td,tderr,nnrs1 = {},{},{},{},{},{}
dfiles['eureka'] = {}
dfiles['eureka']['10nm'] = 'data/Eureka/S6_TOI2031b_Eureka_Transmission_Spectrum_10nm_bins_v1.csv'
dfiles['eureka']['50nm'] = 'data/Eureka/S6_TOI2031b_Eureka_Transmission_Spectrum_50nm_bins_v1.csv'
dfiles['eureka']['20nm'] = 'data/Eureka/S6_TOI2031b_Eureka_Transmission_Spectrum_20nm_bins_v1.csv'
dfiles['tiberius'] = {}
dfiles['tiberius']['10nm'] = 'data/Tiberius/TOI-2031b_Tiberius-JK_spectrum_10nm_Nov25_depths_v1.txt'
dfiles['tiberius']['50nm'] = 'data/Tiberius/TOI-2031b_Tiberius-JK_spectrum_50nm_Nov25_depths_v1.txt'
dfiles['tiberius']['20nm'] = 'data/Tiberius/TOI-2031b_Tiberius-JK_spectrum_20nm_Nov25_depths_v1.txt'
dfiles['tswift'] = {}
dfiles['tswift']['10nm'] = 'data/Tswift/TOI-2031b_Guangwei_spectrum_binned_10nm_Nov10th_v1.csv'
dfiles['tswift']['50nm'] = 'data/Tswift/TOI-2031b_Guangwei_spectrum_binned_50nm_Nov10th_v1.csv'
dfiles['tswift']['20nm'] = 'data/Tswift/TOI-2031b_Guangwei_spectrum_binned_20nm_Nov10th_v1.csv'
wv = {x:{y:{} for y in dfiles[x]} for x in dfiles}
wvbins = {x:{y:{} for y in dfiles[x]} for x in dfiles}
td = {x:{y:{} for y in dfiles[x]} for x in dfiles}
tderr = {x:{y:{} for y in dfiles[x]} for x in dfiles}
nnrs1 = {x:{y:{} for y in dfiles[x]} for x in dfiles}

for x in dfiles:
    if x == 'eureka':
        for y in dfiles[x]:
            wv[x][y],wvbins[x][y],td[x][y],tderr[x][y],nnrs1[x][y] = readInDataEur(dfiles[x][y])
    if x == 'tiberius':
        for y in dfiles[x]:
            wv[x][y],wvbins[x][y],td[x][y],tderr[x][y],nnrs1[x][y] = readInDataTib(dfiles[x][y])
    if x == 'tswift':
        for y in dfiles[x]:
            wv[x][y],wvbins[x][y],td[x][y],tderr[x][y],nnrs1[x][y] = readInDataTsw(dfiles[x][y])


In [ ]:
# Make models 

calculator = TransitDepthCalculator()

Rs=1.241 * R_sun
Mp=0.8 * M_jup 
Rp=1.248 * R_jup
#Rp=89669032.97
T=1358.

modwav_1x, modtd_1x, info_dict_1x = calculator.compute_depths(Rs, Mp, Rp, T, logZ=0, CO_ratio=0.59, 
                                                            full_output=True)
modwav_10x, modtd_10x, info_dict_10x = calculator.compute_depths(Rs, Mp, Rp, T, logZ=1, CO_ratio=0.59, 
                                                            full_output=True)

In [ ]:
# Plot

# Plot settings
plt.rc('font', family='sans-serif')
fontname = 'sans-serif'
labelpad=5

# from toi260.01.mplstyle

plt.rc('lines', linewidth=2.7)              
plt.rc('lines', markerfacecolor='w')
plt.rc('lines', markeredgewidth=2.0)          
plt.rc('lines', markersize=8)  

plt.rc('font', size=17)

plt.rc('text', usetex=False)

plt.rc('axes', labelsize = 17)
plt.rc('axes', linewidth=2)
plt.rc('axes', labelweight='normal')
plt.rc('axes', axisbelow=False)

plt.rc('xtick', top=True)     
plt.rc('xtick', bottom=True)     
plt.rc('xtick.major', size=6)
plt.rc('xtick.minor', size=4)      
plt.rc('xtick.major', width=2)     
plt.rc('xtick.minor', width=2)    
plt.rc('xtick', labelsize=14)      
plt.rc('xtick', direction='in')      
plt.rc('xtick.minor', visible=True)    
plt.rc('xtick.major', top=True)    
plt.rc('xtick.major', bottom=True)    
plt.rc('xtick.minor', top=True)    
plt.rc('xtick.minor', bottom=True)   

plt.rc('ytick', left=True)     
plt.rc('ytick', right=True)     
plt.rc('ytick.major', size=6)
plt.rc('ytick.minor', size=4)      
plt.rc('ytick.major', width=2)     
plt.rc('ytick.minor', width=2)    
plt.rc('ytick', labelsize=14)      
plt.rc('ytick', direction='in')      
plt.rc('ytick.minor', visible=True)    
plt.rc('ytick.major', left=True)    
plt.rc('ytick.major', right=True)    
plt.rc('ytick.minor', left=True)    
plt.rc('ytick.minor', right=True)  
  
plt.rc('legend', fontsize=18)

plt.rc('image', aspect='auto')        
plt.rc('image', origin='lower')        

plt.rc('errorbar', capsize=0)

plt.rc('savefig', bbox='tight')

fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (10,5), facecolor='w')
ax.errorbar(wv['eureka']['50nm']*1e6,td['eureka']['50nm']*1e2,yerr=tderr['eureka']['50nm']*1e2,
            ls='',color='k',marker='o',zorder=0)
ax.set_xlabel('Wavelength ($\mu$m)')
ax.set_ylabel('Transit Depth (%)')  
ax.set_ylim([1.1,1.16])
plt.savefig('fig_eureka_50nm.png', format = 'png')
#plt.savefig('fig_models_toi402.02_final2.pdf', format = 'pdf')
plt.close() 

fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (10,5), facecolor='w')
ax.errorbar(wv['eureka']['50nm']*1e6,td['eureka']['50nm']*1e2,yerr=tderr['eureka']['50nm']*1e2,
            ls='',color='k',marker='o',zorder=0)

binwv = np.linspace(np.min(wv['eureka']['50nm']),np.max(wv['eureka']['50nm']),num=100)    
bintd = spectres(binwv, modwav_1x, modtd_1x)

ax.plot(binwv*1e6,bintd*1e2,color='DeepSkyBlue')
ax.set_xlabel('Wavelength ($\mu$m)')
ax.set_ylabel('Transit Depth (%)')  
#ax.set_ylim([550,1050])
plt.savefig('fig_eureka_50nm_1xsolmod.png', format = 'png')
#plt.savefig('fig_models_toi402.02_final2.pdf', format = 'pdf')
plt.close() 

fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (10,5), facecolor='w')
ax.errorbar(wv['eureka']['50nm']*1e6,td['eureka']['50nm']*1e2,yerr=tderr['eureka']['50nm']*1e2,
            ls='',color='k',marker='o',zorder=0)
ax.plot(binwv*1e6,bintd*1e2,color='DeepSkyBlue')
bintd = spectres(binwv, modwav_10x, modtd_10x)
ax.plot(binwv*1e6,(bintd-0.00017)*1e2,color='Tomato')
ax.set_xlabel('Wavelength ($\mu$m)')
ax.set_ylabel('Transit Depth (%)')  
#ax.set_ylim([550,1050])
plt.savefig('fig_eureka_50nm_10xsolmod.png', format = 'png')
#plt.savefig('fig_models_toi402.02_final2.pdf', format = 'pdf')
plt.close() 

In [ ]:
fname = 'retrieval_results/toi2031b_nlive1000_eqchem_hetlimb_eureka50nm.pkl'
with open(fname,'rb') as f:
    out = pickle.load(f)
print(out.keys())
print(out['labels'])
print(out['best_fit_params'])

In [ ]:
spectout = {}
numsamp = min(1000,len(out['equal_samples'][:,0]))
#wavelengths_list,depths_list,info_dict_list,binwv_list,bintd_list = [],[],[],[],[]
depths_list,bintd_list = [],[]
for i in range(numsamp):
    print('Regenerating spectra',i)
    params = out['equal_samples'][i,:]
    datawv = out['transit_wavelengths']
    #vmrs=[]
    #for j in range(len(species)-1):
    #    vmrs.append(10.**params[j])
    #vmrs.append(1.-np.sum(vmrs))
    Rs = params[0]
    Mp = params[1]
    Rp = params[2]
    T = params[3]
    logZ = params[6]
    CO_ratio = params[9]
    wavelengths, depths, info_dict = calculator.compute_depths(Rs, Mp, Rp, T, logZ=logZ, CO_ratio=CO_ratio, 
                                                               #gases=species,vmrs=vmrs, 
                                                               scattering_factor=10.**params[4],
                                                               scattering_slope=params[5],
                                                               cloudtop_pressure=10.**params[7],
                                                               T_star=6490.,full_output=True)
    binwv = np.linspace(np.min(datawv),np.max(datawv),num=200)    
    bintd = spectres(binwv, wavelengths, depths)

    if i%50 == 0:
        plt.plot(wavelengths*1e6,depths)
        plt.plot(binwv*1e6,bintd)
        plt.plot(out['transit_wavelengths']*1e6,out['random_transit_depths'][i])
        plt.xlim([0.5,5.5])
        plt.show()        

    #wavelengths_list.append(wavelengths)
    depths_list.append(depths)
    #info_dict_list.append(info_dict)
    #binwv_list.append(binwv)
    bintd_list.append(bintd)

spectout['wavelengths'] = wavelengths
spectout['depths'] = depths_list
#spectout['info_dict'] = info_dict_list
spectout['binwv'] = binwv
spectout['bintd'] = bintd_list

f = open('toi2031b_eureka50nm_200PointRandomModelSpectra.pkl','wb')
pickle.dump(spectout,f)
f.close()

In [ ]:
fname = 'toi2031b_eureka50nm_200PointRandomModelSpectra.pkl'
with open(fname,'rb') as f:
    spec = pickle.load(f)

In [ ]:
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (10,5), facecolor='w')
ax.errorbar(wv['eureka']['50nm']*1e6,td['eureka']['50nm']*1e2,yerr=tderr['eureka']['50nm']*1e2,
            ls='',color='k',marker='o',zorder=3)
ax.set_xlabel('Wavelength ($\mu$m)')
ax.set_ylabel('Transit Depth (%)')  
#ax.set_ylim([550,1050])
ax.set_ylim([1.1,1.16])


lower_spectrum_1sig = np.percentile(spec['bintd'], 15.865, axis=0)
upper_spectrum_1sig = np.percentile(spec['bintd'], 84.135, axis=0)
lower_spectrum_2sig = np.percentile(spec['bintd'], 2.275, axis=0)
upper_spectrum_2sig = np.percentile(spec['bintd'], 97.725, axis=0)
lower_spectrum_3sig = np.percentile(spec['bintd'], 0.135, axis=0)
upper_spectrum_3sig = np.percentile(spec['bintd'], 99.865, axis=0)


ax.fill_between(spec['binwv']*1e6,
                 lower_spectrum_3sig*1e2,
                 upper_spectrum_3sig*1e2,
                 color='xkcd:azure',zorder=0,alpha=0.2)
ax.fill_between(spec['binwv']*1e6,
                 lower_spectrum_2sig*1e2,
                 upper_spectrum_2sig*1e2,
                 color='xkcd:azure',zorder=1,alpha=0.5)
ax.fill_between(spec['binwv']*1e6,
                 lower_spectrum_1sig*1e2,
                 upper_spectrum_1sig*1e2,
                 color='xkcd:azure',zorder=2,alpha=1)

plt.savefig('fig_eureka_50nm_retrieval.png', format = 'png')
#plt.savefig('fig_models_toi402.02_final2.pdf', format = 'pdf')
plt.close() 



In [ ]:
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (5,5), facecolor='w')


ct = az.plot_kde(out['equal_samples'][:,6],out['equal_samples'][:,9],
                 hdi_probs=[0.393, 0.865, 0.989],ax=ax,
                 contourf_kwargs={'colors': 'xkcd:azure', 'alpha': 0.3},
                 contour_kwargs={'colors': 'xkcd:azure'})

#ct = az.plot_kde(np.log10(mettot5me/M0),ctoo5me,
#                 hdi_probs=[0.393, 0.865, 0.989],ax=ax2,
#                 contourf_kwargs={'colors': colors[ii], 'alpha': 0.3},
#                 contour_kwargs={'colors': colors[ii]}) 

#ax2.annotate(annotes[ii],xy=(1,0.5),xytext=(-0.7,0.45-ii*0.05),color=colors[ii],horizontalalignment='left') 

ax.axhline(0.59,color='gray',linestyle='--')
ax.axvline(0,color='gray',linestyle='--')
#ax2.axhline(0.59,color='gray',linestyle='--')

#ax1.set_ylim([0,0.65])
#ax1.set_xlim([-1.5,3.5])

#ax1.annotate('Outside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 
#ax2.annotate('Inside 2:1 Mass Prior',xy=(0.5,0.5),xytext=(0.5,1.04),color='k',xycoords='axes fraction',
#             verticalalignment='center',horizontalalignment='center') 

#ax1.annotate('Solar C/O = 0.59 (Asplund et al. 2021)',xy=(0,0.5),xytext=(1,0.56),color='gray',
#             horizontalalignment='center') 
#ax2.annotate('Solar C/O = 0.59 (Asplund et al. 2021)',xy=(0,0.5),xytext=(1,0.56),color='gray',
#             horizontalalignment='center') 

ax.set_ylabel('C/O')
ax.set_xlabel('[M/H]')

plt.savefig('fig_metc2oposteriors.png', format = 'png')
plt.close()

In [ ]:
spectouttls = {}
numsamp = min(1000,len(outtls['equal_samples'][:,0]))
#wavelengths_list,depths_list,info_dict_list,binwv_list,bintd_list = [],[],[],[],[]
depths_list,bintd_list = [],[]
for i in range(numsamp):
    print('Regenerating spectra',i)
    params = outtls['equal_samples'][i,:]
    datawv = outtls['transit_wavelengths']
    #vmrs=[]
    #for j in range(len(species)-1):
    #    vmrs.append(10.**params[j])
    #vmrs.append(1.-np.sum(vmrs))
    Rs = 0.82*R_sun
    Mp = 10.*M_earth
    Rp = params[2]
    T = 600.0
    logZ = -1.
    CO_ratio = 0.59
    wavelengths, depths, info_dict = calculator.compute_depths(Rs, Mp, Rp, T, logZ=logZ, CO_ratio=CO_ratio, 
                                                               #gases=species,vmrs=vmrs, 
                                                               scattering_factor=1.,
                                                               scattering_slope=4.,
                                                               cloudtop_pressure=10.**-3.999,
                                                               T_star=params[0],#T_spot=params[1],
                                                               #spot_cov_frac=params[3],
                                                               full_output=True)
        
    depths[0:len(wavelengths[wavelengths<datawv[n102-1]])] += params[5]
    
    binwv = np.linspace(np.min(datawv),np.max(datawv),num=100)    
    bintd = spectres(binwv, wavelengths, depths)

    if i%50 == 0:
        plt.plot(wavelengths*1e6,depths)
        plt.plot(binwv*1e6,bintd)
        plt.plot(outtls['transit_wavelengths']*1e6,outtls['random_transit_depths'][i])
        plt.xlim([0.5,2.0])
        plt.show()        

    #wavelengths_list.append(wavelengths)
    depths_list.append(depths)
    #info_dict_list.append(info_dict)
    #binwv_list.append(binwv)
    bintd_list.append(bintd)

spectouttls['wavelengths'] = wavelengths
spectouttls['depths'] = depths_list
#spectouttls['info_dict'] = info_dict_list
spectouttls['binwv'] = binwv
spectouttls['bintd'] = bintd_list